In [1]:
using ITensors
using ITensorMPS
using Random
using LinearAlgebra
using Statistics
using PrettyTables
#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end
#= 
    A collection of functions to generate 
    to generate random initial product states  
    using the random_mps method. 
    
    These states corresponds to diferent 
    phases of matter and are used according 
    to the phase diagram of the EHM.
=#

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end 
function random_sdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i=1:2:(L-Nup_extra)
        state[i] = "Up"
        state[i+1] = "Dn"
    end
    if Nup != 0 
        state[L] = "Up"
    end
    return state 
end
function random_ps_state(L, Nup, Ndn)
    state = fill("Emp", L)
    
    Ndbl = min(Nup, Ndn)
    Nup -= Ndbl
    Ndn -= Ndbl
    
    for i in 1:Ndbl
        state[i] = "UpDn"
    end
    
    next_site = Ndbl + 1
    
    if Nup > 0
        state[next_site] = "Up"
        next_site += 1
    elseif Ndn > 0
        state[next_site] = "Dn"
        next_site += 1
    end
    return state
end
#= 
    Returns a state for the point (U, V) of the Extended Hubbard Model.
=#
function state_ehm_diagram(L, Nup, Ndn, U, V)

    state = fill("Emp", L)
    
    region = get_region(U, V)

    # Weak coupling = metallic
    if region == "METALLIC" 
        state = random_metallic_state(L, Nup, Ndn)
    # CDW
    elseif region == "CDW"
        state = random_cdw_state(L, Nup, Ndn)
    elseif region == "SDW"
        state = random_sdw_state(L, Nup, Ndn)
    else
        state = random_ps_state(L, Nup, Ndn)
    end
    return state
end
#= 
    Trying to separate the phase-diagram states not only 
    in big squared blocks.
    See notebook ploting_ehm_diagram_selection_of_states.ipynb for an example of plot 
    using this function. I try to made it look like the plot schematic 
    diagram for the EHM. 
=#
function get_region(u, v)

    alpha = 0.5

    # Boundary functions
    get_metallic_lower_boundary(u) = u <= 0 ? -exp(alpha * u) : -alpha * u - 1.0
    get_metallic_upper_boundary_h(u) = -0.1 * u
    get_metallic_upper_boundary_negative(u) = -2.5 * alpha * u

    lower = get_metallic_lower_boundary(u)
    upper_h = get_metallic_upper_boundary_h(u)
    upper_neg = get_metallic_upper_boundary_negative(u)

    if v <= lower
        return "PS"
    elseif (u < 0 && v < 0 && v > lower) ||
           (u > 0 && v > 0 && v < upper_h) ||
           (u > 0 && v < 0 && v > lower && v < upper_neg)
        return "METALLIC"
    elseif (u < 0 && v > 0) ||
           (u > 0 && v > 0 && v >= 2 * u)
        return "CDW"
    elseif (u > 0 && v < 0 && v > lower) ||
           (u > 0 && v > 0 && v >= upper_h && v < 2 * u)
        return "SDW"
    end
    return "PS"
end
#=
    Create the basis for pairs of spins 
    1 = (1, up) , 2 = (1, dn), 3 = (2, up), ...
=#
function two_fermion_basis_pairs(L)
    num_modes = 2 * L
    pairs = []
    for m1 in 1:num_modes
        for m2 in (m1 + 1):num_modes
            push!(pairs, (m1, m2))
        end
    end
    return pairs
end
#=
    Gets the indexed site and spin.
=#
function mode_to_site_spin(m::Int)
    site = (m + 1) ÷ 2
    spin = isodd(m) ? "up" : "dn"
    return site, spin
end
#= 
    Returns a two-particle reduced density matrix. 

    This function computes only the part of the matrix in which 
    all of the correlators are different. 
    
    Tolerance of the julia language packages are very low, so in general
    this computation gives various non-hermitian matrices. 
=#
function build_2_particle_rdm(phi, sites)

    L = length(phi)
    
    # Basis for pairs of fermions.
    pairs = two_fermion_basis_pairs(L)

    dim = length(pairs)

    # @show dim

    rho_2 = zeros(ComplexF64, dim, dim)
    #= 
        Loops through all the configurations with no repeated indices and spins, 
        stores the elements rho_2[p, q] = <psi' | O | psi>. 
    =#
    for (p, (i, j)) in enumerate(pairs)
        # @show (p, (i, j))

        i_site, i_spin = mode_to_site_spin(i)
        j_site, j_spin = mode_to_site_spin(j)

        # @show (mode_to_site_spin(i), mode_to_site_spin(j))

        for (q, (k, l)) in enumerate(pairs)
            # @show (q, (k, l))
           
            os = OpSum() 

            k_site, k_spin = mode_to_site_spin(k)
            l_site, l_spin = mode_to_site_spin(l)

            # @show (mode_to_site_spin(k), mode_to_site_spin(l))

            os -= "Cdag"*i_spin, i_site, "Cdag"*j_spin, j_site, "C"*k_spin, k_site, "C"*l_spin, l_site

            O_mpo = MPO(os, sites)

            rho_2[p, q] = inner(phi', O_mpo, phi)
        end
    end
    # Force hermiticity
    # rho_2 = (rho_2 + rho_2') / 2.0
    # Normalize by remaining number of particles 
    # rho_2 = (2.0 / (L*(L-1))) * rho_2
    return rho_2
end

build_2_particle_rdm (generic function with 1 method)

In [2]:
#=
    Create the basis for pairs of spins 
    1 = (1, up) , 2 = (1, dn), 3 = (2, up), ...
=#
function two_fermion_basis_pairs(L)
    num_modes = 2 * L
    pairs = []
    for m1 in 1:num_modes
        for m2 in (m1 + 1):num_modes
            push!(pairs, (m1, m2))
        end
    end
    return pairs
end
#=
    Gets the indexed site and spin.
=#
function mode_to_site_spin(m::Int)
    site = (m + 1) ÷ 2
    spin = isodd(m) ? "up" : "dn"
    return site, spin
end

mode_to_site_spin (generic function with 1 method)

In [ ]:
L = 4

Npart = floor(Int, L/2)
Nup = Npart + L % 2
Ndn = L - Nup

J = 1.0
U = 0.0
V = 4.0

sites = siteinds("Electron", L; conserve_qns=true)

H = H_EHM(L, J, U, V, sites)
#= 
    The best state for variational step of the DMRG algorithm.
=#
state = state_ehm_diagram(L, Nup, Ndn, U, V)
# dmrg parameters 
nsweeps = 6
m = 10
maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

psi0 = random_mps(sites, state; linkdims=m)
# Start DMRG calculation:
energy, psi = dmrg(H, psi0; nsweeps, maxdim=maxdim, cutoff=cutoff)

pairs = two_fermion_basis_pairs(L)
counter=0
for (p, (i, j)) in enumerate(pairs)

    i_site, i_spin = mode_to_site_spin(i)
    j_site, j_spin = mode_to_site_spin(j)

    for (q, (k, l)) in enumerate(pairs)

        os = OpSum() 
        k_site, k_spin = mode_to_site_spin(k)
        l_site, l_spin = mode_to_site_spin(l)

        os -= "Cdag"*i_spin, i_site, "Cdag"*j_spin, j_site, "C"*k_spin, k_site, "C"*l_spin, l_site

        O_mpo = MPO(os, sites)

        sign_amp = inner(psi', O_mpo, psi)
        
        result = abs(sign_amp) < 1e-12 ? 0.0 : (sign_amp > 0 ? 1.0 : -1.0)

        # if (inner(psi', O_mpo, psi) != 0)
        println(counter, '\t', "Cdag$i Cdag$j C$k C$l",
        '\t', i, ' ', j, ' ',  k, ' ', l, '\t', '\t', re]sult, '\t', sign_amp)
        # end
        counter+=1
    end
end

After sweep 1 energy=-0.6704479287717977  maxlinkdim=16 maxerr=0.00E+00 time=0.011
After sweep 2 energy=-1.0220692991238356  maxlinkdim=16 maxerr=0.00E+00 time=0.017
After sweep 3 energy=-1.333366890127533  maxlinkdim=16 maxerr=0.00E+00 time=0.013
After sweep 4 energy=-1.365174687828145  maxlinkdim=16 maxerr=0.00E+00 time=0.018
After sweep 5 energy=-1.3668788479212968  maxlinkdim=16 maxerr=0.00E+00 time=0.012
After sweep 6 energy=-1.3669663026492422  maxlinkdim=16 maxerr=0.00E+00 time=0.017
0	Cdag1 Cdag2 C1 C2	1 2 1 2		1.0	0.6991858355177933
1	Cdag1 Cdag2 C1 C3	1 2 1 3		0.0	0.0
2	Cdag1 Cdag2 C1 C4	1 2 1 4		1.0	0.1764118963631349
3	Cdag1 Cdag2 C1 C5	1 2 1 5		0.0	0.0
4	Cdag1 Cdag2 C1 C6	1 2 1 6		1.0	0.01744676979215408
5	Cdag1 Cdag2 C1 C7	1 2 1 7		0.0	0.0
6	Cdag1 Cdag2 C1 C8	1 2 1 8		-1.0	-0.01718203520476084
7	Cdag1 Cdag2 C2 C3	1 2 2 3		-1.0	-0.17644628076016608
8	Cdag1 Cdag2 C2 C4	1 2 2 4		0.0	0.0
9	Cdag1 Cdag2 C2 C5	1 2 2 5		-1.0	-0.017363276458838114
10	Cdag1 Cdag2 C2 C6	1 2 2 6		0.0

In [ ]:
for L in [2, 4, 5, 6, 7, 8, 9, 10, 11, 12]

    Npart = floor(Int, L/2)
    Nup = Npart + L % 2
    Ndn = L - Nup

    J = 1.0
    U = 0.0
    V = 4.0

    sites = siteinds("Electron", L; conserve_qns=true)

    H = H_EHM(L, J, U, V, sites)
    #= 
    The best state for variational step of the DMRG algorithm.
    =#
    state = state_ehm_diagram(L, Nup, Ndn, U, V)
    # dmrg parameters 
    nsweeps = 6
    m = 10
    maxdim = [50, 100, 200, 400, 800, 800]
    cutoff = [1E-14]

    psi0 = random_mps(sites, state; linkdims=m)
    # Start DMRG calculation:
    energy, psi = dmrg(H, psi0; nsweeps, maxdim=maxdim, cutoff=cutoff)

    pairs = two_fermion_basis_pairs(L)
    
    filename = "fermionic_signs_EHM_L=$(L).txt"

    open(filename, "w") do io
        counter=0
        for (p, (i, j)) in enumerate(pairs)

            i_site, i_spin = mode_to_site_spin(i)
            j_site, j_spin = mode_to_site_spin(j)

            for (q, (k, l)) in enumerate(pairs)

                os = OpSum() 
                k_site, k_spin = mode_to_site_spin(k)
                l_site, l_spin = mode_to_site_spin(l)

                os -= "Cdag"*i_spin, i_site, "Cdag"*j_spin, j_site, "C"*k_spin, k_site, "C"*l_spin, l_site

                O_mpo = MPO(os, sites)

                sign_amp = inner(psi', O_mpo, psi)
                
                result = abs(sign_amp) < 1e-12 ? 0.0 : (sign_amp > 0 ? 1.0 : -1.0)

                println(counter, '\t', result, '\t', sign_amp)

                println(io, result)
            end
        end
    end
end

┌ Warning: MPS center bond dimension is less than requested (you requested 10, but in practice it is 4. This is likely due to technicalities of truncating quantum number sectors.
└ @ ITensorMPS /home/jefter66/.julia/packages/ITensorMPS/3SBBV/src/mps.jl:153


After sweep 1 energy=-0.8272782383624688  maxlinkdim=4 maxerr=0.00E+00 time=0.001
After sweep 2 energy=-0.8284271247038176  maxlinkdim=4 maxerr=0.00E+00 time=0.001
After sweep 3 energy=-0.82842712474619  maxlinkdim=4 maxerr=0.00E+00 time=0.001
After sweep 4 energy=-0.8284271247461902  maxlinkdim=4 maxerr=0.00E+00 time=0.001
After sweep 5 energy=-0.8284271247461902  maxlinkdim=4 maxerr=0.00E+00 time=0.000
After sweep 6 energy=-0.8284271247461902  maxlinkdim=4 maxerr=0.00E+00 time=0.000
0	1.0	0.4267766952966334
0	0.0	0.0
0	1.0	0.17677669529663617
0	-1.0	-0.1767766952966362
0	0.0	0.0
0	1.0	0.42677669529663687
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	1.0	0.17677669529663617
0	0.0	0.0
0	1.0	0.07322330470336312
0	-1.0	-0.07322330470336313
0	0.0	0.0
0	1.0	0.1767766952966376
0	-1.0	-0.1767766952966362
0	0.0	0.0
0	-1.0	-0.07322330470336313
0	1.0	0.07322330470336316
0	0.0	0.0
0	-1.0	-0.17677669529663761
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	0.0	0.0
0	1.0	0.4267

Excessive output truncated after 524301 bytes.

0.0	0.0
0	-1.0	-0.0013567391988980428
0	0.0	0.0
0	1.0	0.0019071992615902434
0	0.0	0.0
0	1.0	0.017927276094955265
0	0.0	0.0
0	1.0	0.03694529663135896
0	0.0	0.0
0	1.0	0.003773526471456164
0	0.0	0.0
0	1.0	0.0006030447002514156
0	1.0	0.0014334353803525269
0	0.0	0.0
0	1.0	0.0005793549902282217
0	0.0	0.0
0	1.0	5.7343679959367914e-5
0	0.0	0.0
0	-1.0	-3.6937774254565255e-5
0	0.0	0.0
0	-1.0	